In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib
import os

In [2]:
def generate_synthetic_data(n=1000, seed=42):
    np.random.seed(seed)
    df = pd.DataFrame({
        'weekly_work_hours': np.clip(np.random.normal(40, 10, n), 20, 70),
        'overtime_hours': np.clip(np.random.exponential(3, n), 0, 20),
        'meeting_density': np.clip(np.random.beta(2, 5, n), 0, 0.9),
        'consecutive_work_days': np.random.randint(1, 15, n),
        'conflict_count': np.random.poisson(1.5, n),
        'workload_trend': np.random.normal(0, 0.3, n)
    })

    df['night_weekend_hours'] = df['overtime_hours'] * np.random.uniform(0.2, 0.6, n)

    X1 = np.clip(df['overtime_hours'] / 10, 0, 1)
    X2 = np.clip(df['meeting_density'] / 0.7, 0, 1)
    X3 = np.clip(df['consecutive_work_days'] / 14, 0, 1)
    X4 = np.clip(df['conflict_count'] / 5, 0, 1)
    X5 = np.clip(df['workload_trend'] * 2 + 0.5, 0, 1)
    df['risk_score_raw'] = 10 * (0.35*X1 + 0.25*X2 + 0.20*X3 + 0.15*X4 + 0.05*X5)
    df['risk_score'] = np.clip(df['risk_score_raw'] + np.random.normal(0, 0.4, n), 0, 10)
    return df

In [7]:
def train_and_save():
    df = generate_synthetic_data()
    features = ['weekly_work_hours', 'overtime_hours', 'meeting_density',
                'consecutive_work_days', 'conflict_count', 'workload_trend', 'night_weekend_hours']
    X = df[features]
    y = df['risk_score']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = GradientBoostingRegressor(n_estimators=150, max_depth=4, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)

    mae = mean_absolute_error(y_test, model.predict(X_test))

    os.makedirs('models', exist_ok=True)
    joblib.dump(model, 'models/risk_model.pkl')
    return model

if __name__ == '__main__':
    train_and_save()